# Modelling Quantum Fluids with the Gross-Pitaevskii Equation

This notebook runs a 2D **Gross-Pitaevskii Equation (GPE)** simulator (`GrossPitaevskiiEquation2D`). This equation models the dynamics of Bose-Einstein Condensates (BECs) and superfluid helium (quantum fluids).

- **State variables**: `density` ($|\psi|^2$), `real` ($\Re(\psi)$), and `imag` ($\Im(\psi)$).
- **Conditioning variables**: `g` controls non-linear interactions, while `disorder_strength`, `spoon_strength`, `wx`, and `wy` adjust the trap potential geometry and complexities.
- **Dynamics**: Split-Step Fourier Method alternating between real (potential and interaction) and momentum (kinetic) space.
- **Boundary conditions**: Naturally periodic via the Fast Fourier Transform (FFT) combined with localized harmonic trapping.

### Why this is useful

In classical fluid dynamics datasets, models learn to predict pressure and velocity. In quantum mechanics:

- **Density** acts analogously to "pressure/mass".
- **Real/imag** provide a smooth representation of the wavefunction and avoid branch-cut discontinuities from wrapped phase.
- **Phase** is still recoverable as `atan2(imag, real)` when needed.

By manipulating the complexity knobs in the potential trap (like adding a "laser spoon" to stir the fluid or "optical speckle" for spatial disorder), we can generate anywhere from simple harmonic oscillations to chaotic quantum turbulence.


## Governing equations

The Gross-Pitaevskii Equation describes the ground state of a quantum system of identical bosons using a pseudo-wavefunction $\psi(x,y,t)$.

$$
i \hbar \frac{\partial \psi}{\partial t} = \left( -\frac{\hbar^2}{2m}\nabla^2 + V(r) + g |\psi|^2  \right) \psi
$$

Where:

- $\nabla^2$ is the kinetic energy operator.
- $V(r)$ is the external trapping potential.
- $g|\psi|^2$ represents the non-linear inter-particle interaction (e.g. atomic repulsion).

In this simulator, we use dimensionless units where $\hbar = 1$ and $m = 1$. The wave function is stepped forward in time using the **Split-Step Fourier Method** due to its unconditional numerical stability and probability conservation compared to standard ODE solvers.


### Simulation Parameters Explained

In both the low and high complexity examples, we configure parameters that control physically meaningful regimes and dataset diversity:

- **`g`**: Non-linear interaction strength. $g>0$ models repulsive bosons; $g=0$ recovers the linear Schrödinger limit.
- **`disorder_strength`**: Amplitude of a spatially random, speckle-like potential (kept stationary by default for physical consistency).
- **`spoon_strength`**: Amplitude of a localized Gaussian obstacle ("laser spoon").
- **`spoon_speed`**: Angular speed of the spoon trajectory in the lab-frame potential.
- **`Omega`**: Rotating-frame angular velocity term ($-\Omega L_z$), independent of spoon motion.
- **`wx`, `wy`**: Harmonic trap frequencies; larger values produce tighter confinement.
- **`x0`, `y0`**: Initial condensate center.
- **`kx0`, `ky0`**: Initial momentum kick.
- **`width`**: Initial Gaussian width (smaller values induce stronger breathing).
- **`imaginary_time`**: If enabled, evolves in imaginary time to relax toward low-energy states (with renormalization each step).

### Physical modeling scope

This simulator is a mean-field $T\approx 0$ model (Gross-Pitaevskii Equation). External laser/disorder effects are represented as classical potentials in $V(x,y,t)$. It does **not** explicitly model decoherence, thermal clouds, or stochastic dissipation; those require extensions such as SGPE/ZNG-type models.

In [ ]:
from IPython.display import HTML

from autosim.experimental.simulations import GrossPitaevskiiEquation2D as GPESim
from autosim.utils import plot_spatiotemporal_video


### Low Complexity Example

When $g=0$, the system reduces to the linear Schrödinger equation. With a perfectly round harmonic trap (`wx=1.0, wy=1.0`), the wave behaves highly predictably.

If we place a Gaussian wavepacket inside this trap, its behavior depends heavily on its initial `width`:

- **Coherent state (`width=1.0`)**: If the initial width perfectly matches the trap's natural length scale ($1/\sqrt{w_x} = 1.0$), it acts exactly like a classical ball on a spring. It will slosh back and forth forever without its shape changing or spreading out.
- **Squeezed state (`width=0.4`)**: By squishing the initial wavepacket so its `width` is thinner than the trap's natural scale, we force it to disperse (spread out) as it moves. Because it is trapped, it bounces off the edges and folds back in on itself.

**What you will see:**

1. **Sloshing:** The center of the wavepacket swings side-to-side in the trap.
2. **Breathing:** The wavepacket alternately expands (spreads out) and contracts (squeezes back together) as it moves.
3. **Interference fringes:** As the spreading wavepacket hits the trap walls and reflects back over itself, the overlapping waves create high-contrast ripples (constructive and destructive interference).


In [ ]:
sim_easy = GPESim(
    return_timeseries=True,
    log_level="progress_bar",
    n=128,  # grid resolution 128x128
    T=24.0,  # 24/3.14 ≈ 7.64 breathing periods, ~4 full periods (T_trap = 2π ≈ 6.28)
    dt=0.005,
    snapshot_dt=0.1,  # finer snapshots to capture the breathing clearly
    L=10.0,
    parameters_range={
        "g": (0.0, 0.0),
        "disorder_strength": (0.0, 0.0),
        "spoon_strength": (0.0, 0.0),
        "wx": (1.0, 1.0),
        "wy": (1.0, 1.0),
        # Matplotlib imshow displays the mathematical Y-axis vertically by default.
        # Simulation creates grids using ij indexing, meaning X is row (vertical).
        # Push the packet along the y-axis making it slosh purely horizontally visually.
        "x0": (0.0, 0.0),
        "y0": (1.5, 1.5),
        "kx0": (0.0, 0.0),
        "ky0": (-2.0, -2.0),
        # width < a_ho = 1/sqrt(wx) = 1.0 => squeezed state: packet breathes and
        # spreads, producing visible interference fringes as it overlaps with itself
        "width": (0.4, 0.4),
    },
    random_seed=42,
)

res_easy = sim_easy.forward_samples_spatiotemporal(n=1)

print("Constant Scalars (Batch x Params):", res_easy["constant_scalars"].shape)
print(
    "Spatiotemporal Output:",
    res_easy["data"].shape,
    "[batch, time, x, y, channels={density, real, imag}]",
)


In [ ]:
anim_easy = plot_spatiotemporal_video(
    res_easy["data"],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
)
HTML(anim_easy.to_jshtml())

### High Complexity / Turbulence Example

The simulation parameters can be tuned to produce strongly non-linear, vortex-rich dynamics useful for ML forecasting benchmarks.

- **Strong Repulsion (`g=50`)**: Increases nonlinear pressure and suppresses simple overlap dynamics.
- **Disorder (`disorder_strength=0.8`)**: Static speckle-like landscape that scatters and fragments flow.
- **Laser Spoon (`spoon_strength=3.0`, `spoon_speed=1.5`)**: A moving obstacle that injects energy and sheds wakes/vortices.
- **Rotating Frame (`Omega=0.2`)**: Adds rotational bias through the $-\Omega L_z$ term.
- **Tighter Trap (`wx=2.0`, `wy=2.0`)**: Raises central density and strengthens nonlinear interactions.

**What you will see:**

1. **Wake formation and shear:** Flow wraps around the moving obstacle and forms complex trailing structures.
2. **Density fragmentation:** Speckle scattering and stirring produce intermittent low/high-density patches.
3. **Wavefunction winding:** Vortex cores appear where density dips and the reconstructed phase (`atan2(imag, real)`) winds by $2\pi$ around the core.

In [ ]:
Omega = 0.2
spoon_speed = 3.0
trap_strength = 2.0
sim_hard = GPESim(
    return_timeseries=True,
    log_level="progress_bar",
    n=128,
    T=20.0,
    dt=0.005,
    snapshot_dt=0.3,
    L=10.0,
    parameters_range={
        "g": (50.0, 50.0),
        "disorder_strength": (0.8, 0.8),
        "spoon_strength": (3.0, 3.0),
        "spoon_speed": (spoon_speed, spoon_speed),
        "wx": (trap_strength, trap_strength),
        "wy": (trap_strength, trap_strength),
        "x0": (0.0, 0.0),
        "kx0": (0.0, 0.0),
        "Omega": (Omega, Omega),
        "imaginary_time": (0, 0),
        "box_param": (0.1, 0.1)
    },
    random_seed=42,
)

res_hard = sim_hard.forward_samples_spatiotemporal(n=1)

In [ ]:
anim_hard = plot_spatiotemporal_video(
    res_hard["data"],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
)
HTML(anim_hard.to_jshtml())

In [ ]:
# Numerical stability diagnostics
import torch

density = res_hard["data"][0, ..., 0]
print("density min/max:", float(density.min()), float(density.max()))
print("density p95/p99:", float(torch.quantile(density, 0.95)), float(torch.quantile(density, 0.99)))
print("density max first/last:", float(density[0].max()), float(density[-1].max()))
print("nan/inf:", int(torch.isnan(density).sum()), int(torch.isinf(density).sum()))


## Dataset Generation

We generate four dataset families on a `64 x 64` grid with `321` time points (`T=16.0`, `snapshot_dt=0.05`, so $16/0.05=320$ intervals + initial state).

Each family targets a different dynamical regime for model robustness:

1. **`low_complexity`**: Mild interactions, no spoon, no disorder; mostly regular oscillatory behavior.
2. **`laser_only`**: Moving obstacle drives wake/vortex dynamics without static disorder.
3. **`speckle_only`**: Stationary disordered potential induces scattering and fragmentation.
4. **`laser_and_speckle`**: Combined forcing/scattering for the richest spatiotemporal complexity.

A `quick_demo` switch is used below so the same notebook supports both fast local checks and larger training-scale generation.

In [ ]:
# T=16.0, snapshot_dt=0.05 -> 16 / 0.05 = 320 steps. 320 + 1 (initial) = 321 time steps
T_dataset = 16.0
dt = 0.005
snapshot_dt = 0.05
n_grid = 64

quick_demo = True
if quick_demo:
    split_sizes = {"train": 4, "valid": 2, "test": 2}
else:
    split_sizes = {"train": 200, "valid": 20, "test": 20}

# Base parameter dictionary template with light diversity
base_params = {
    "wx": (1.0, 1.0),
    "wy": (1.0, 1.0),
    "box_param": (0.1, 0.1),
    "width": (0.4, 0.8),
    "x0": (-1.0, 1.0),
    "y0": (-1.0, 1.0),
    "kx0": (-2.0, 2.0),
    "ky0": (-2.0, 2.0),
    "imaginary_time": (0, 0),
}

dataset_configs = {
    "low_complexity": {
        **base_params,
        "g": (0.0, 5.0),
        "disorder_strength": (0.0, 0.0),
        "spoon_strength": (0.0, 0.0),
        "spoon_speed": (0.0, 0.0),
        "Omega": (0.0, 0.0),
    },
    "laser_only": {
        **base_params,
        "g": (30.0, 50.0),
        "disorder_strength": (0.0, 0.0),
        "spoon_strength": (2.0, 4.0),
        "spoon_speed": (1.0, 2.0),
        "Omega": (0.1, 0.4),
    },
    "speckle_only": {
        **base_params,
        "g": (30.0, 50.0),
        "disorder_strength": (0.5, 1.2),
        "spoon_strength": (0.0, 0.0),
        "spoon_speed": (0.0, 0.0),
        "Omega": (0.0, 0.0),
    },
    "laser_and_speckle": {
        **base_params,
        "g": (30.0, 50.0),
        "disorder_strength": (0.5, 1.2),
        "spoon_strength": (2.0, 4.0),
        "spoon_speed": (1.0, 2.0),
        "Omega": (0.1, 0.4),
    },
}

outputs = {}

for name, params in dataset_configs.items():
    print(f"Generating dataset: {name}")
    sim = GPESim(
        return_timeseries=True,
        log_level="warning",
        n=n_grid,
        T=T_dataset,
        dt=dt,
        snapshot_dt=snapshot_dt,
        L=10.0,
        parameters_range=params,
        random_seed=42,
    )

    outputs[name] = {}
    for split, count in split_sizes.items():
        print(f"  -> Split: {split} (n={count})")
        res = sim.forward_samples_spatiotemporal(n=count)
        outputs[name][split] = res

print("Finished generating all datasets!")
print(
    "Example shape for low_complexity train:",
    outputs["low_complexity"]["train"]["data"].shape,
)

In [ ]:
from pathlib import Path

import torch

# Save datasets to disk
output_root = Path("./generated_datasets")
for name in dataset_configs:
    for split in ["train", "valid", "test"]:
        save_dir = output_root / f"gpe_{name}" / split
        save_dir.mkdir(parents=True, exist_ok=True)
        output_path = save_dir / "data.pt"
        torch.save(outputs[name][split], output_path)

print(f"Saved all datasets under: {output_root.resolve()}")

In [ ]:
anim = plot_spatiotemporal_video(
    outputs["low_complexity"]["train"]["data"],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True
)
HTML(anim.to_jshtml())

In [ ]:
anim = plot_spatiotemporal_video(
    outputs["laser_only"]["train"]["data"],
    batch_idx=0,
    channel_names=["density", "real", "imag"],
)
HTML(anim.to_jshtml())

In [ ]:
anim = plot_spatiotemporal_video(
    outputs["speckle_only"]["train"]["data"],
    batch_idx=3,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True
)
HTML(anim.to_jshtml())

In [ ]:
anim = plot_spatiotemporal_video(
    outputs["laser_and_speckle"]["train"]["data"],
    batch_idx=2,
    channel_names=["density", "real", "imag"],
    preserve_aspect=True
)
HTML(anim.to_jshtml())